# 43 — FIM Critical Exponents: Does the Strange Loop Change Universality?

**UNPRECEDENTED:** Standard Kuramoto has mean-field critical exponents:
β = 1/2 (R ~ (K-K_c)^β), γ = 1 (susceptibility), ν = 1/2 (correlation).
SPO validated β = 1/2 for classical Kuramoto (V4, V52).

Does FIM change the universality class? NB26 Binder cumulant → 2/3
(mean-field), suggesting it doesn't. But we haven't measured β directly.

If β ≠ 1/2 under FIM → new universality class → publishable.
If β = 1/2 → FIM preserves universality despite being self-referential.

## Method
1. Finite-size scaling: R(K, N) at K near K_c for N = {8, 12, 16, 20}
2. Extract β from R ~ (K - K_c)^β
3. Extract γ from χ = d<R²>/dK ~ |K - K_c|^(-γ)
4. Compare with/without FIM

In [ ]:
import numpy as np
import json
from pathlib import Path
from scipy.optimize import curve_fit

import sys
sys.path.insert(0, '/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src')
from scpn_quantum_control.bridge.knm_hamiltonian import build_knm_paper27, OMEGA_N_16

def fim_gradient_all(phases):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases + np.pi) % (2 * np.pi) - np.pi
    sensitivity = min(1.0 / (1.0 - R**2 + 0.01), 50.0)
    return (1.0 / n) * np.sin(phase_diff) * sensitivity

def simulate_R_stats(N, K_scale, lam, dt=0.02, T=200.0, noise=0.05, n_seeds=5):
    """Return R mean, R², R⁴ for Binder cumulant and susceptibility."""
    K = build_knm_paper27(L=min(N, 16))
    if N > 16:
        K_full = np.zeros((N, N))
        K_full[:16, :16] = K
        for i in range(16, N):
            for j in range(N):
                if i != j: K_full[i, j] = K_full[j, i] = 0.45 * np.exp(-0.3 * abs(i-j))
        K = K_full
    K = K * K_scale
    omega = OMEGA_N_16[:min(N, 16)]
    if N > 16:
        rng_o = np.random.default_rng(12345)
        omega = np.concatenate([omega, rng_o.uniform(omega.min(), omega.max(), N-16)])
    
    R_all = []
    for seed in range(n_seeds):
        rng = np.random.default_rng(seed)
        theta = rng.uniform(0, 2*np.pi, N)
        n_steps = int(T / dt)
        R_tail = []
        for s in range(n_steps):
            diff = theta[None, :] - theta[:, None]
            coupling = np.sum(K * np.sin(diff), axis=1) / N
            dphi = omega + coupling
            if lam > 0: dphi += lam * fim_gradient_all(theta)
            theta = (theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)) % (2*np.pi)
            if s >= n_steps * 3 // 4:
                R_tail.append(float(np.abs(np.mean(np.exp(1j * theta)))))
        R_all.extend(R_tail)
    
    R_arr = np.array(R_all)
    return {
        'R': float(np.mean(R_arr)),
        'R2': float(np.mean(R_arr**2)),
        'R4': float(np.mean(R_arr**4)),
        'chi': float(np.var(R_arr) * N),  # susceptibility
    }

print('Ready.')

In [ ]:
# Fine K sweep near K_c for different N, with and without FIM
N_values = [8, 12, 16, 20]

# First pass: find approximate K_c for each (N, lambda)
for lam in [0.0, 1.0]:
    print(f'\n=== lambda = {lam} ===')
    
    for N in N_values:
        # Coarse sweep to find K_c
        K_coarse = np.linspace(2, 22, 11)
        R_coarse = []
        for K_s in K_coarse:
            stats = simulate_R_stats(N, K_s, lam, n_seeds=2)
            R_coarse.append(stats['R'])
        
        # K_c where R crosses 0.5
        R_arr = np.array(R_coarse)
        K_c_est = None
        for i in range(len(R_arr) - 1):
            if R_arr[i] < 0.5 <= R_arr[i+1]:
                frac = (0.5 - R_arr[i]) / (R_arr[i+1] - R_arr[i] + 1e-10)
                K_c_est = K_coarse[i] + frac * (K_coarse[i+1] - K_coarse[i])
                break
        if K_c_est is None:
            K_c_est = K_coarse[np.argmin(np.abs(R_arr - 0.5))]
        
        print(f'N={N:2d}: K_c ≈ {K_c_est:.1f}')
        
        # Fine sweep: 15 points in [K_c-3, K_c+5]
        K_fine = np.linspace(max(0.5, K_c_est - 3), K_c_est + 5, 15)
        R_fine = []
        chi_fine = []
        for K_s in K_fine:
            stats = simulate_R_stats(N, K_s, lam, n_seeds=3)
            R_fine.append(stats['R'])
            chi_fine.append(stats['chi'])
        
        # Fit beta: R = A * (K - K_c)^beta for K > K_c
        above = [(K_fine[i], R_fine[i]) for i in range(len(K_fine)) if R_fine[i] > 0.3 and K_fine[i] > K_c_est]
        if len(above) >= 3:
            K_above = np.array([a[0] for a in above]) - K_c_est
            R_above = np.array([a[1] for a in above])
            try:
                def power(x, A, beta):
                    return A * np.power(np.maximum(x, 0.01), beta)
                popt, _ = curve_fit(power, K_above, R_above, p0=[0.5, 0.5], maxfev=5000)
                print(f'      β = {popt[1]:.3f} (mean-field: 0.500)')
            except Exception as e:
                print(f'      β fit failed: {str(e)[:40]}')
        
        # Susceptibility peak
        chi_peak = max(chi_fine)
        K_chi_peak = K_fine[np.argmax(chi_fine)]
        print(f'      χ_max = {chi_peak:.2f} at K = {K_chi_peak:.1f}')

In [ ]:
# Finite-size scaling: χ_max ~ N^(gamma/nu)
print('\n=== FINITE-SIZE SCALING ===')
for lam in [0.0, 1.0]:
    print(f'\nλ = {lam}:')
    chi_max_vs_N = []
    for N in N_values:
        K_sweep = np.linspace(5, 20, 10)
        chi_best = 0
        for K_s in K_sweep:
            stats = simulate_R_stats(N, K_s, lam, n_seeds=3)
            chi_best = max(chi_best, stats['chi'])
        chi_max_vs_N.append(chi_best)
        print(f'  N={N:2d}: χ_max = {chi_best:.2f}')
    
    # Fit χ_max = a * N^(gamma/nu)
    N_arr = np.array(N_values, dtype=float)
    chi_arr = np.array(chi_max_vs_N)
    if all(c > 0 for c in chi_arr):
        try:
            log_N = np.log(N_arr)
            log_chi = np.log(chi_arr)
            slope, intercept = np.polyfit(log_N, log_chi, 1)
            print(f'  γ/ν = {slope:.3f} (mean-field: γ/ν = 2.0)')
        except:
            print('  Fit failed.')

print('\n=== VERDICT ===')
print('If β ≈ 0.5 with and without FIM → FIM preserves mean-field universality.')
print('If β differs → FIM creates a new universality class.')

In [ ]:
# Save
output = {'experiment': 'FIM critical exponents — universality class', 'N_values': N_values}
out_path = Path('/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/critical_exponents_2026-03-29.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Results saved to {out_path}')